In [2]:
import sys
import torch
import pandas as pd
from ultralytics import YOLO

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
print("FoodVision AI V2 - READY!")

Python: 3.11.15
PyTorch: 2.15.0+cpu
CUDA: False
FoodVision AI V2 - READY!


In [3]:
from pathlib import Path

PROJECT = Path(r"F:\AI SAMSUNG\FoodVision-AI-V2")
DATASETS = PROJECT / "datasets"

print("Project:", PROJECT)
print("Project tồn tại:", PROJECT.exists())
print("Datasets tồn tại:", DATASETS.exists())

Project: F:\AI SAMSUNG\FoodVision-AI-V2
Project tồn tại: True
Datasets tồn tại: True


In [4]:
from pathlib import Path

OLD_PROJECT = Path(r"F:\AI SAMSUNG\FoodVision-AI")
OLD_DATASETS = OLD_PROJECT / "datasets"

print("Dự án cũ tồn tại:", OLD_PROJECT.exists())
print("\nCÁC DATASET HIỆN CÓ:")

if OLD_DATASETS.exists():
    for item in sorted(OLD_DATASETS.iterdir()):
        if item.is_dir():
            print(f"[FOLDER] {item.name}")
        elif item.is_file():
            print(f"[FILE]   {item.name}")
else:
    print("Không tìm thấy thư mục dữ liệu cũ.")

Dự án cũ tồn tại: True

CÁC DATASET HIỆN CÓ:
[FOLDER] A_Food_214
[FOLDER] B_Raw_Detection
[FOLDER] C_Freshness
[FOLDER] D_New_Raw


In [5]:
from pathlib import Path

OLD_MODELS = Path(r"F:\AI SAMSUNG\FoodVision-AI\models")

if OLD_MODELS.exists():
    for model in sorted(OLD_MODELS.rglob("*")):
        if model.is_file() and model.suffix.lower() in [".pt", ".pth"]:
            size_mb = model.stat().st_size / (1024 * 1024)
            print(f"{model.name:35} {size_mb:.2f} MB")
else:
    print("Không tìm thấy thư mục models cũ.")

best_original.pt                    49.86 MB
freshness_beef.pth                  15.56 MB
freshness_chicken.pth               15.56 MB
freshness_fish.pth                  15.56 MB
freshness_shrimp.pth                15.56 MB
yolo_raw_best.pt                    6.16 MB


In [1]:
from pathlib import Path
import yaml

ROOT = Path(r"F:\AI SAMSUNG\FoodVision-AI-V2")
DATA_DIR = ROOT / "datasets" / "food_cooked"

yaml_files = list(DATA_DIR.rglob("data.yaml"))

if not yaml_files:
    raise FileNotFoundError("Không tìm thấy data.yaml!")

DATA_YAML = yaml_files[0]

with open(DATA_YAML, "r", encoding="utf-8") as f:
    config = yaml.safe_load(f)

print("Dataset:", DATA_YAML)
print("Số lớp:", config.get("nc"))
print("Train:", config.get("train"))
print("Validation:", config.get("val"))
print("Test:", config.get("test"))


Dataset: F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_cooked\data.yaml
Số lớp: 214
Train: ../train/images
Validation: ../valid/images
Test: ../test/images


In [2]:
from pathlib import Path

DATA_ROOT = DATA_YAML.parent

for split in ["train", "valid", "test"]:
    folder = DATA_ROOT / split / "images"

    if folder.exists():
        images = [
            p for p in folder.iterdir()
            if p.suffix.lower() in [".jpg", ".jpeg", ".png"]
        ]
        print(f"{split}: {len(images):,} ảnh")
    else:
        print(f"{split}: Không thấy thư mục {folder}")

train: 32,723 ảnh
valid: 9,339 ảnh
test: 4,673 ảnh


In [3]:
from pathlib import Path
import yaml

ROOT = Path(r"F:\AI SAMSUNG\FoodVision-AI-V2")
DATA_DIR = ROOT / "datasets" / "food_cooked"

yaml_files = list(DATA_DIR.rglob("data.yaml"))
assert len(yaml_files) == 1, f"Cần đúng 1 data.yaml, tìm thấy: {yaml_files}"

DATA_YAML = yaml_files[0]
cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))

print("Dataset:", DATA_YAML)
print("Số lớp:", cfg.get("nc", len(cfg.get("names", []))))
print("Train:", cfg.get("train"))
print("Val:", cfg.get("val"))
print("Test:", cfg.get("test"))

assert len(cfg.get("names", [])) == 214, "Danh sách lớp không đúng 214!"

Dataset: F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_cooked\data.yaml
Số lớp: 214
Train: ../train/images
Val: ../valid/images
Test: ../test/images


In [5]:
from pathlib import Path
import yaml

ROOT = Path(r"F:\AI SAMSUNG\FoodVision-AI-V2")

# DATA_YAML đã được tìm thấy ở ô code trước
DATA_ROOT = DATA_YAML.parent

# Tạo cấu hình YOLO chính xác
clean_cfg = {
    "path": str(DATA_ROOT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "names": cfg["names"],
    "nc": 214
}

# Kiểm tra đường dẫn thực tế
for split in ["train", "val", "test"]:
    folder = DATA_ROOT / clean_cfg[split]
    assert folder.is_dir(), f"Không tìm thấy: {folder}"
    print(f"{split}: OK - {folder}")

# Lưu cấu hình mới, không sửa file gốc
TRAIN_YAML = ROOT / "datasets" / "food_cooked_train.yaml"

with open(TRAIN_YAML, "w", encoding="utf-8") as f:
    yaml.safe_dump(
        clean_cfg,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print("\nĐã tạo cấu hình YOLO thành công!")
print("File:", TRAIN_YAML)

train: OK - F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_cooked\train\images
val: OK - F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_cooked\valid\images
test: OK - F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_cooked\test\images

Đã tạo cấu hình YOLO thành công!
File: F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_cooked_train.yaml


In [6]:
from ultralytics import YOLO
import time

model = YOLO("yolov8n.pt")

start = time.time()

results = model.train(
    data=str(TRAIN_YAML),
    epochs=20,
    fraction=1.0,
    imgsz=640,
    batch=4,
    device="cpu",
    workers=0,
    cache=False,
    patience=20,
    save=True,
    save_period=1,
    val=True,
    plots=True,
    project=str(ROOT / "models" / "yolo_food"),
    name="food_214_train20",
    exist_ok=False
)

print(f"Thời gian: {(time.time() - start) / 3600:.2f} giờ")

Ultralytics 8.4.191  Python-3.11.15 torch-2.15.0+cpu CPU (13th Gen Intel Core i7-1355U)
engine\trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=4, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_cooked_train.yaml, degrees=0.0, deterministic=True, device=cpu, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=20, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=0.7, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=food_214_train20, nbs

OSError: [WinError 433] A device which does not exist was specified: 'F:\\AI SAMSUNG\\FoodVision-AI-V2\\datasets\\food_cooked\\train\\images\\11254_jpg.rf.10bbc52df39e69ee9f4bfc11155e730e.npy'

In [1]:
from pathlib import Path
import yaml

ROOT = Path(r"F:\AI SAMSUNG\FoodVision-AI-V2")

DATASET = (
    ROOT / "datasets" / "food_raw"
    / "source" / "food_ingredients"
)

assert DATASET.exists(), f"Không tìm thấy dataset: {DATASET}"

yaml_files = list(DATASET.rglob("data.yaml"))

print("Đường dẫn dataset:", DATASET)
print("Số file data.yaml:", len(yaml_files))

for file in yaml_files:
    print("Tìm thấy:", file)

assert len(yaml_files) == 1, (
    "Cần kiểm tra lại vị trí data.yaml!"
)

DATA_YAML = yaml_files[0]
cfg = yaml.safe_load(
    DATA_YAML.read_text(encoding="utf-8")
)

names = cfg["names"]

if isinstance(names, dict):
    classes = [names[k] for k in sorted(names, key=int)]
else:
    classes = list(names)

print("\nTỔNG SỐ LỚP:", len(classes))

for i, name in enumerate(classes):
    print(f"{i:02d} - {name}")

AssertionError: Không tìm thấy dataset: F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_raw\source\food_ingredients

In [2]:
from pathlib import Path
import yaml

ROOT = Path(r"F:\AI SAMSUNG\FoodVision-AI-V2")

DATASET = ROOT / "datasets" / "food_raw" / "source"
DATA_YAML = DATASET / "data.yaml"

assert DATA_YAML.is_file(), "Không tìm thấy data.yaml"

with open(DATA_YAML, "r", encoding="utf-8") as f:
    cfg = yaml.safe_load(f)

names = cfg["names"]

if isinstance(names, dict):
    classes = [names[k] for k in sorted(names, key=int)]
else:
    classes = list(names)

print("TỔNG SỐ LỚP:", len(classes))
print("\nDANH SÁCH LỚP:")

for i, name in enumerate(classes):
    print(f"{i:02d} - {name}")

TỔNG SỐ LỚP: 20

DANH SÁCH LỚP:
00 - beef
01 - bell_pepper
02 - cabbage
03 - carrot
04 - cauliflower
05 - chicken
06 - cucumber
07 - egg
08 - fish
09 - garlic
10 - ginger
11 - kumquat
12 - lemon
13 - onion
14 - pork
15 - potato
16 - shrimp
17 - small_pepper
18 - tofu
19 - tomato


In [3]:
from pathlib import Path

for split in ["train", "valid", "test"]:
    images_dir = DATASET / split / "images"
    labels_dir = DATASET / split / "labels"

    images = list(images_dir.glob("*")) if images_dir.exists() else []
    labels = list(labels_dir.glob("*.txt")) if labels_dir.exists() else []

    print(f"{split}: {len(images):,} ảnh | {len(labels):,} file nhãn")

train: 35,823 ảnh | 35,823 file nhãn
valid: 2,442 ảnh | 2,442 file nhãn
test: 2,157 ảnh | 2,157 file nhãn


In [ ]:
from pathlib import Path
from collections import Counter
import shutil
import yaml

ROOT = Path(r"F:\AI SAMSUNG\FoodVision-AI-V2")
SOURCE = ROOT / "datasets" / "food_raw" / "source"
OUTPUT = ROOT / "datasets" / "food_raw" / "processed" / "raw_5classes_v1"

# Class ID cũ -> Class ID mới
CLASS_MAP = {
    16: 0,  # shrimp
    8: 1,   # fish
    5: 2,   # chicken
    14: 3,  # pork
    0: 4    # beef
}

NEW_NAMES = [
    "raw_shrimp",
    "raw_fish",
    "raw_chicken",
    "raw_pork",
    "raw_beef"
]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

# Không ghi đè dataset đã xử lý trước đó
if OUTPUT.exists():
    raise FileExistsError(
        f"Dataset đầu ra đã tồn tại: {OUTPUT}\n"
        "Hãy kiểm tra trước khi tạo phiên bản mới."
    )

stats = {}

for split in ["train", "valid", "test"]:
    image_dir = SOURCE / split / "images"
    label_dir = SOURCE / split / "labels"

    if not image_dir.is_dir() or not label_dir.is_dir():
        raise FileNotFoundError(
            f"Thiếu images hoặc labels trong {split}"
        )

    out_images = OUTPUT / split / "images"
    out_labels = OUTPUT / split / "labels"

    out_images.mkdir(parents=True, exist_ok=True)
    out_labels.mkdir(parents=True, exist_ok=True)

    total_images = 0
    kept_images = 0
    class_counts = Counter()

    for image_path in image_dir.iterdir():
        if image_path.suffix.lower() not in IMAGE_EXTS:
            continue

        total_images += 1
        label_path = label_dir / f"{image_path.stem}.txt"

        if not label_path.exists():
            continue

        new_lines = []

        for line in label_path.read_text(
            encoding="utf-8"
        ).splitlines():

            parts = line.strip().split()

            if len(parts) != 5:
                continue

            try:
                old_id = int(parts[0])
                coords = [float(v) for v in parts[1:]]
            except ValueError:
                continue

            if old_id not in CLASS_MAP:
                continue

            if not (
                0 <= coords[0] <= 1
                and 0 <= coords[1] <= 1
                and 0 < coords[2] <= 1
                and 0 < coords[3] <= 1
                and coords[0] - coords[2] / 2 >= 0
                and coords[1] - coords[3] / 2 >= 0
                and coords[0] + coords[2] / 2 <= 1
                and coords[1] + coords[3] / 2 <= 1
            ):
                continue

            new_id = CLASS_MAP[old_id]

            new_lines.append(
                f"{new_id} " +
                " ".join(parts[1:])
            )

            class_counts[NEW_NAMES[new_id]] += 1

        # Chỉ giữ ảnh chứa ít nhất một nguyên liệu mục tiêu
        if not new_lines:
            continue

        shutil.copy2(image_path, out_images / image_path.name)

        (out_labels / f"{image_path.stem}.txt").write_text(
            "\n".join(new_lines) + "\n",
            encoding="utf-8"
        )

        kept_images += 1

    stats[split] = {
        "total": total_images,
        "kept": kept_images,
        "boxes": dict(class_counts)
    }

# Tạo data.yaml mới
data_config = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 5,
    "names": NEW_NAMES
}

with open(OUTPUT / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(
        data_config,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print("ĐÃ TẠO DATASET YOLO RAW 5 LỚP")
print("Đường dẫn:", OUTPUT)

for split, info in stats.items():
    print(f"\n{split.upper()}")
    print("Ảnh nguồn:", info["total"])
    print("Ảnh được giữ:", info["kept"])
    print("Số bounding boxes theo lớp:")

    for name in NEW_NAMES:
        print(f"  {name}: {info['boxes'].get(name, 0)}")

In [ ]:
from pathlib import Path
from collections import Counter
from tqdm.auto import tqdm
import shutil
import yaml

ROOT = Path(r"F:\AI SAMSUNG\FoodVision-AI-V2")

SOURCE = ROOT / "datasets" / "food_raw" / "source"

OUTPUT = (
    ROOT / "datasets" / "food_raw" /
    "processed" / "raw_5classes_v2"
)

# Mapping 20 lớp nguồn -> 5 lớp nguyên liệu
CLASS_MAP = {
    16: 0,  # shrimp
    8: 1,   # fish
    5: 2,   # chicken
    14: 3,  # pork
    0: 4    # beef
}

NAMES = [
    "raw_shrimp",
    "raw_fish",
    "raw_chicken",
    "raw_pork",
    "raw_beef"
]

EXTENSIONS = {".jpg", ".jpeg", ".png", ".webp"}

# Không ghi đè một bản dữ liệu đang tồn tại
if OUTPUT.exists():
    raise FileExistsError(
        f"Đã tồn tại: {OUTPUT}. "
        "Hãy chọn tên phiên bản mới nếu cần chạy lại."
    )

# Kiểm tra đầy đủ nguồn trước khi bắt đầu
for split in ["train", "valid", "test"]:
    assert (SOURCE / split / "images").is_dir(), f"Thiếu ảnh {split}"
    assert (SOURCE / split / "labels").is_dir(), f"Thiếu nhãn {split}"

summary = {}

for split in ["train", "valid", "test"]:

    image_dir = SOURCE / split / "images"
    label_dir = SOURCE / split / "labels"

    output_images = OUTPUT / split / "images"
    output_labels = OUTPUT / split / "labels"

    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)

    image_files = sorted(
        p for p in image_dir.iterdir()
        if p.is_file() and p.suffix.lower() in EXTENSIONS
    )

    counters = Counter()
    kept = 0
    invalid = 0
    mixed = 0

    for image_path in tqdm(image_files, desc=f"Filtering {split}"):

        label_path = label_dir / (image_path.stem + ".txt")

        if not label_path.is_file():
            continue

        new_labels = []
        has_other_classes = False

        for line in label_path.read_text(
            encoding="utf-8"
        ).splitlines():

            fields = line.strip().split()

            if len(fields) != 5:
                invalid += 1
                continue

            try:
                old_class = int(fields[0])
                x, y, w, h = map(float, fields[1:])
            except ValueError:
                invalid += 1
                continue

            if not (
                0 <= x <= 1 and 0 <= y <= 1
                and 0 < w <= 1 and 0 < h <= 1
                and x - w / 2 >= -1e-5
                and y - h / 2 >= -1e-5
                and x + w / 2 <= 1 + 1e-5
                and y + h / 2 <= 1 + 1e-5
            ):
                invalid += 1
                continue

            if old_class not in CLASS_MAP:
                has_other_classes = True
                continue

            new_id = CLASS_MAP[old_class]

            new_labels.append(
                f"{new_id} {x} {y} {w} {h}"
            )

        if not new_labels:
            continue

        # Bỏ ảnh chứa cả lớp cần giữ và lớp ngoài phạm vi,
        # tránh tạo đối tượng thiếu nhãn
        if has_other_classes:
            mixed += 1
            continue

        shutil.copy2(
            image_path,
            output_images / image_path.name
        )

        (output_labels / (image_path.stem + ".txt")).write_text(
            "\n".join(new_labels) + "\n",
            encoding="utf-8"
        )

        kept += 1

        for label in new_labels:
            counters[int(label.split()[0])] += 1

    summary[split] = {
        "images_source": len(image_files),
        "images_kept": kept,
        "mixed_skipped": mixed,
        "invalid_lines": invalid,
        "classes": dict(counters)
    }

    print(
        f"\n{split.upper()}: "
        f"{kept:,}/{len(image_files):,} ảnh được giữ",
        flush=True
    )

# Tạo data.yaml sau khi hoàn thành các split
data_cfg = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 5,
    "names": NAMES
}

with open(OUTPUT / "data.yaml", "w", encoding="utf-8") as f:
    yaml.safe_dump(
        data_cfg,
        f,
        allow_unicode=True,
        sort_keys=False
    )

print("\nHOÀN THÀNH DATASET YOLO RAW 5 LỚP!")

for split, data in summary.items():
    print(f"\n{split.upper()}:")
    print("Ảnh nguồn:", data["images_source"])
    print("Ảnh giữ lại:", data["images_kept"])
    print("Ảnh hỗn hợp bỏ qua:", data["mixed_skipped"])
    print("Dòng nhãn lỗi:", data["invalid_lines"])

    for class_id, name in enumerate(NAMES):
        print(
            f"  {name}: "
            f"{data['classes'].get(class_id, 0):,} boxes"
        )

print("\nDataset mới:", OUTPUT)

D:\AI-SAMSUNG\envs\sic_foodvision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [6]:
%pip install tqdm

  Using cached tqdm-4.70.1-py3-none-any.whl.metadata (57 kB)
Using cached tqdm-4.70.1-py3-none-any.whl (80 kB)
Note: you may need to restart the kernel to use updated packages.


In [ ]:
%pip install ipywidgets

In [1]:
from tqdm.auto import tqdm

for i in tqdm(range(100), desc="Kiem tra tien do"):
    pass

print("Thanh tien do hoat dong!")

D:\AI-SAMSUNG\envs\sic_foodvision\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Kiem tra tien do: 100%|██████████| 100/100 [00:00<?, ?it/s]

Thanh tien do hoat dong!


In [2]:

from pathlib import Path
from collections import Counter
from tqdm import tqdm
import shutil
import yaml

# =====================================================
# 1. CAU HINH DUONG DAN
# =====================================================

ROOT = Path(r"F:\AI SAMSUNG\FoodVision-AI-V2")

SOURCE = ROOT / "datasets" / "food_raw" / "source"

OUTPUT = (
    ROOT / "datasets" / "food_raw" /
    "processed" / "raw_5classes_v3"
)

# =====================================================
# 2. CHUYEN NHAN 20 LOP -> 5 LOP
# =====================================================

CLASS_MAP = {
    16: 0,   # shrimp
    8: 1,    # fish
    5: 2,    # chicken
    14: 3,   # pork
    0: 4     # beef
}

EXPECTED_SOURCE_NAMES = {
    16: "shrimp",
    8: "fish",
    5: "chicken",
    14: "pork",
    0: "beef"
}

NEW_NAMES = [
    "raw_shrimp",
    "raw_fish",
    "raw_chicken",
    "raw_pork",
    "raw_beef"
]

IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".webp"}

# =====================================================
# 3. KIEM TRA DU LIEU NGUON
# =====================================================

source_yaml = SOURCE / "data.yaml"

assert source_yaml.is_file(), (
    f"Khong tim thay data.yaml: {source_yaml}"
)

with source_yaml.open("r", encoding="utf-8") as f:
    source_cfg = yaml.safe_load(f)

source_names = source_cfg["names"]

if isinstance(source_names, dict):
    source_names = {
        int(k): str(v)
        for k, v in source_names.items()
    }
else:
    source_names = dict(enumerate(source_names))

assert len(source_names) == 20, (
    f"Dataset co {len(source_names)} lop, khong phai 20!"
)

for old_id, expected_name in EXPECTED_SOURCE_NAMES.items():
    actual_name = source_names.get(old_id)
    assert actual_name == expected_name, (
        f"ID {old_id}: can '{expected_name}', "
        f"nhung data.yaml la '{actual_name}'"
    )

for split in ["train", "valid", "test"]:
    assert (SOURCE / split / "images").is_dir(), (
        f"Thieu thu muc images cua {split}"
    )
    assert (SOURCE / split / "labels").is_dir(), (
        f"Thieu thu muc labels cua {split}"
    )

if OUTPUT.exists():
    raise FileExistsError(
        f"Thu muc dau ra da ton tai:\n{OUTPUT}\n"
        "Hay chon ten phien ban moi de tranh ghi de."
    )

print("KIEM TRA DATASET THANH CONG!")
print("Nguon:", SOURCE)
print("Dau ra:", OUTPUT)

# =====================================================
# 4. LOC ANH VA CHUYEN NHAN
# =====================================================

summary = {}

for split in ["train", "valid", "test"]:

    source_images = SOURCE / split / "images"
    source_labels = SOURCE / split / "labels"

    output_images = OUTPUT / split / "images"
    output_labels = OUTPUT / split / "labels"

    output_images.mkdir(parents=True, exist_ok=True)
    output_labels.mkdir(parents=True, exist_ok=True)

    image_files = sorted([
        p for p in source_images.iterdir()
        if p.is_file() and p.suffix.lower() in IMAGE_EXTS
    ])

    box_counts = Counter()

    kept_images = 0
    skipped_mixed = 0
    skipped_no_target = 0
    skipped_invalid = 0
    missing_labels = 0

    print(f"\nDANG XU LY {split.upper()}")
    print("Tong so anh:", len(image_files), flush=True)

    for image_path in tqdm(
        image_files,
        desc=f"Filtering {split}"
    ):

        label_path = (
            source_labels / f"{image_path.stem}.txt"
        )

        if not label_path.is_file():
            missing_labels += 1
            continue

        try:
            lines = label_path.read_text(
                encoding="utf-8"
            ).splitlines()
        except OSError as exc:
            raise RuntimeError(
                f"Khong doc duoc nhan {label_path}. "
                "Kiem tra ket noi o F:."
            ) from exc

        converted_labels = []
        contains_other_classes = False
        invalid_image_labels = False

        for line in lines:

            fields = line.strip().split()

            if not fields:
                continue

            if len(fields) != 5:
                invalid_image_labels = True
                break

            try:
                old_id = int(fields[0])
                x, y, w, h = map(float, fields[1:])
            except ValueError:
                invalid_image_labels = True
                break

            if not (
                0 <= x <= 1
                and 0 <= y <= 1
                and 0 < w <= 1
                and 0 < h <= 1
                and x - w / 2 >= -1e-5
                and y - h / 2 >= -1e-5
                and x + w / 2 <= 1 + 1e-5
                and y + h / 2 <= 1 + 1e-5
            ):
                invalid_image_labels = True
                break

            if old_id not in source_names:
                invalid_image_labels = True
                break

            if old_id not in CLASS_MAP:
                contains_other_classes = True
                continue

            new_id = CLASS_MAP[old_id]

            converted_labels.append(
                f"{new_id} {x:.8f} {y:.8f} "
                f"{w:.8f} {h:.8f}"
            )

        # Loai anh co nhan loi
        if invalid_image_labels:
            skipped_invalid += 1
            continue

        # Loai anh khong co 5 nguyen lieu muc tieu
        if not converted_labels:
            skipped_no_target += 1
            continue

        # Loai anh chua ca lop muc tieu va lop ngoai pham vi
        # De tranh bo sot nhan doi tuong trong anh
        if contains_other_classes:
            skipped_mixed += 1
            continue

        # Sao chep anh, giu nguyen dataset goc
        try:
            shutil.copy2(
                image_path,
                output_images / image_path.name
            )

            (output_labels / f"{image_path.stem}.txt").write_text(
                "\n".join(converted_labels) + "\n",
                encoding="utf-8"
            )

        except OSError as exc:
            raise RuntimeError(
                f"Loi doc/ghi du lieu tai {image_path}. "
                "Kiem tra USB va o F:."
            ) from exc

        kept_images += 1

        for new_line in converted_labels:
            new_id = int(new_line.split()[0])
            box_counts[new_id] += 1

    summary[split] = {
        "source_images": len(image_files),
        "kept_images": kept_images,
        "skipped_mixed": skipped_mixed,
        "skipped_no_target": skipped_no_target,
        "skipped_invalid": skipped_invalid,
        "missing_labels": missing_labels,
        "boxes": dict(box_counts)
    }

    print(
        f"HOAN THANH {split.upper()}: "
        f"{kept_images:,} anh duoc giu",
        flush=True
    )

# =====================================================
# 5. TAO DATA.YAML MOI
# =====================================================

new_yaml = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 5,
    "names": NEW_NAMES
}

with (OUTPUT / "data.yaml").open(
    "w", encoding="utf-8"
) as f:
    yaml.safe_dump(
        new_yaml,
        f,
        allow_unicode=True,
        sort_keys=False
    )

# =====================================================
# 6. IN KET QUA
# =====================================================

print("\n" + "=" * 55)
print("HOAN THANH LOC DATASET YOLO RAW 5 LOP!")
print("=" * 55)

for split, info in summary.items():

    print(f"\n--- {split.upper()} ---")
    print("Anh nguon:", info["source_images"])
    print("Anh giu lai:", info["kept_images"])
    print("Anh hon hop bo qua:", info["skipped_mixed"])
    print("Anh khong co lop muc tieu:", info["skipped_no_target"])
    print("Anh co nhan loi:", info["skipped_invalid"])
    print("Anh thieu file nhan:", info["missing_labels"])

    for class_id, name in enumerate(NEW_NAMES):
        count = info["boxes"].get(class_id, 0)
        print(f"  {name}: {count:,} bounding boxes")

print("\nDATASET MOI:")
print(OUTPUT)

print("\nFILE CAU HINH:")
print(OUTPUT / "data.yaml")


KIEM TRA DATASET THANH CONG!
Nguon: F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_raw\source
Dau ra: F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_raw\processed\raw_5classes_v3

DANG XU LY TRAIN
Tong so anh: 35823


Filtering train:   0%|          | 3/35823 [00:00<2:10:51,  4.56it/s]


RuntimeError: Loi doc/ghi du lieu tai F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_raw\source\train\images\-11_jpg.rf.468ab9b87529d1706993a63590e6592f.jpg. Kiem tra USB va o F:.

In [3]:
from pathlib import Path

DATA = Path(
    r"F:\AI SAMSUNG\FoodVision-AI-V2"
    r"\datasets\food_raw\processed\raw_5classes_v1"
)

print("=== KIỂM TRA DATASET V1 ===")

print("data.yaml:", (DATA / "data.yaml").exists())

for split in ["train", "valid", "test"]:
    images = DATA / split / "images"
    labels = DATA / split / "labels"

    image_count = sum(
        1 for p in images.glob("*")
        if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
    )

    label_count = len(list(labels.glob("*.txt")))

    print(f"\n{split.upper()}")
    print("Ảnh:", image_count)
    print("Nhãn:", label_count)
    print("Khớp số lượng:", image_count == label_count)

=== KIỂM TRA DATASET V1 ===
data.yaml: False

TRAIN
Ảnh: 7676
Nhãn: 7931
Khớp số lượng: False

VALID
Ảnh: 0
Nhãn: 0
Khớp số lượng: True

TEST
Ảnh: 0
Nhãn: 0
Khớp số lượng: True


In [ ]:

from pathlib import Path
from collections import Counter
import os
import shutil
import math
import yaml
import time

# ======================================================
# 1. DUONG DAN - RESUME V1
# ======================================================

ROOT = Path(r"F:\AI SAMSUNG\FoodVision-AI-V2")

SOURCE = ROOT / "datasets" / "food_raw" / "source"

OUTPUT = (
    ROOT / "datasets" / "food_raw" /
    "processed" / "raw_5classes_v1"
)

CLASS_MAP = {
    16: 0,  # shrimp
    8: 1,   # fish
    5: 2,   # chicken
    14: 3,  # pork
    0: 4    # beef
}

EXPECTED = {
    16: "shrimp",
    8: "fish",
    5: "chicken",
    14: "pork",
    0: "beef"
}

NEW_NAMES = [
    "raw_shrimp",
    "raw_fish",
    "raw_chicken",
    "raw_pork",
    "raw_beef"
]

EXTS = {".jpg", ".jpeg", ".png", ".webp"}
SPLITS = ["train", "valid", "test"]

# ======================================================
# 2. KIEM TRA NGUON
# ======================================================

with (SOURCE / "data.yaml").open(
    "r", encoding="utf-8"
) as f:
    source_cfg = yaml.safe_load(f)

names = source_cfg["names"]

if isinstance(names, list):
    names = dict(enumerate(names))
else:
    names = {int(k): v for k, v in names.items()}

assert len(names) == 20, "Dataset nguon khong du 20 lop"

for old_id, name in EXPECTED.items():
    assert names.get(old_id) == name, (
        f"Sai nhan nguon ID {old_id}"
    )

for split in SPLITS:
    assert (SOURCE / split / "images").is_dir()
    assert (SOURCE / split / "labels").is_dir()

OUTPUT.mkdir(parents=True, exist_ok=True)

# ======================================================
# 3. HAM HO TRO
# ======================================================

def parse_source_label(path):
    """
    Tra ve:
      None neu nhan loi hoac co lop ngoai pham vi
      [] neu khong co nguyen lieu muc tieu
      list neu hop le
    """
    if not path.is_file():
        return None

    converted = []
    has_other = False

    for line in path.read_text(
        encoding="utf-8"
    ).splitlines():

        parts = line.split()

        if not parts:
            continue

        if len(parts) != 5:
            return None

        try:
            old_id = int(parts[0])
            coords = [float(v) for v in parts[1:]]
        except ValueError:
            return None

        x, y, w, h = coords

        if (
            old_id not in names
            or not all(math.isfinite(v) for v in coords)
            or not (0 <= x <= 1 and 0 <= y <= 1)
            or not (0 < w <= 1 and 0 < h <= 1)
            or x - w / 2 < -1e-5
            or y - h / 2 < -1e-5
            or x + w / 2 > 1 + 1e-5
            or y + h / 2 > 1 + 1e-5
        ):
            return None

        if old_id not in CLASS_MAP:
            has_other = True
        else:
            converted.append(
                (CLASS_MAP[old_id], coords)
            )

    # Bo qua anh hon hop de tranh thieu nhan
    if has_other:
        return None

    return converted


def make_label_text(rows):
    return "".join(
        str(class_id) + " " +
        " ".join(f"{v:.8f}" for v in coords) + "\n"
        for class_id, coords in rows
    )


def same_labels(path, expected_rows):
    if not path.is_file():
        return False

    try:
        lines = [
            line.split()
            for line in path.read_text(
                encoding="utf-8"
            ).splitlines()
            if line.strip()
        ]

        if len(lines) != len(expected_rows):
            return False

        for fields, (class_id, coords) in zip(
            lines, expected_rows
        ):
            if len(fields) != 5:
                return False

            if int(fields[0]) != class_id:
                return False

            actual = [float(x) for x in fields[1:]]

            if any(
                not math.isfinite(a)
                or abs(a - b) > 1e-5
                for a, b in zip(actual, coords)
            ):
                return False

        return True

    except (OSError, ValueError):
        return False


def same_image_size(src, dst):
    if not dst.is_file():
        return False

    try:
        if dst.stat().st_size == 0:
            return False

        return (
            src.stat().st_size == dst.stat().st_size
        )
    except OSError:
        return False


def save_image(src, dst):
    # Ghi qua file tam, tranh anh bi cat doan
    temp = dst.with_name(dst.name + ".resume_tmp")

    try:
        if temp.exists():
            temp.unlink()

        try:
            os.link(src, temp)
        except OSError:
            shutil.copy2(src, temp)

        if temp.stat().st_size != src.stat().st_size:
            raise IOError("Kich thuoc anh khong khop")

        os.replace(temp, dst)

    finally:
        if temp.exists():
            temp.unlink()


def save_label(dst, rows):
    temp = dst.with_name(dst.name + ".resume_tmp")

    try:
        temp.write_text(
            make_label_text(rows),
            encoding="utf-8"
        )
        os.replace(temp, dst)

    finally:
        if temp.exists():
            temp.unlink()


# ======================================================
# 4. TIEP TUC XU LY
# ======================================================

start_time = time.time()
summary = {}

print("RESUME DATASET YOLO RAW V1")
print("Source:", SOURCE)
print("Output:", OUTPUT, flush=True)

for split in SPLITS:

    src_img_dir = SOURCE / split / "images"
    src_lab_dir = SOURCE / split / "labels"

    dst_img_dir = OUTPUT / split / "images"
    dst_lab_dir = OUTPUT / split / "labels"

    dst_img_dir.mkdir(parents=True, exist_ok=True)
    dst_lab_dir.mkdir(parents=True, exist_ok=True)

    images = sorted(
        p for p in src_img_dir.iterdir()
        if p.is_file() and p.suffix.lower() in EXTS
    )

    # Cac anh duoc phep ton tai sau khi loc
    expected_images = set()
    expected_labels = set()

    kept = 0
    repaired = 0
    skipped = 0
    boxes = Counter()
    seen_stems = set()

    print(
        f"\nDANG RESUME {split.upper()}: "
        f"{len(images):,} anh nguon",
        flush=True
    )

    for index, img in enumerate(images, start=1):

        if img.stem in seen_stems:
            raise ValueError(
                f"Trung ten anh: {img.stem}"
            )
        seen_stems.add(img.stem)

        src_label = src_lab_dir / (img.stem + ".txt")

        rows = parse_source_label(src_label)

        if rows is None or len(rows) == 0:
            skipped += 1
            continue

        dst_img = dst_img_dir / img.name
        dst_lab = dst_lab_dir / (img.stem + ".txt")

        expected_images.add(img.name)
        expected_labels.add(dst_lab.name)

        image_ok = same_image_size(img, dst_img)
        label_ok = same_labels(dst_lab, rows)

        if not image_ok:
            save_image(img, dst_img)

        if not label_ok:
            save_label(dst_lab, rows)

        if not image_ok or not label_ok:
            repaired += 1

        kept += 1

        for class_id, coords in rows:
            boxes[class_id] += 1

        if index % 500 == 0:
            print(
                f"{split}: {index:,}/{len(images):,} "
                f"| Hop le: {kept:,} "
                f"| Tao/sua: {repaired:,}",
                flush=True
            )

    # Chuyen file du sang thu muc du phong, khong xoa
    quarantine = OUTPUT / "_recovery_orphans" / split
    moved = 0

    for folder, allowed in [
        (dst_img_dir, expected_images),
        (dst_lab_dir, expected_labels)
    ]:
        for file in folder.iterdir():
            if not file.is_file():
                continue

            if file.name in allowed:
                continue

            target_dir = quarantine / folder.name
            target_dir.mkdir(parents=True, exist_ok=True)

            target = target_dir / file.name

            if target.exists():
                raise FileExistsError(
                    f"File du phong da ton tai: {target}"
                )

            shutil.move(str(file), str(target))
            moved += 1

    summary[split] = {
        "images": kept,
        "repaired": repaired,
        "skipped": skipped,
        "moved": moved,
        "boxes": boxes
    }

    print(
        f"HOAN THANH {split.upper()}: "
        f"{kept:,} anh, "
        f"{repaired:,} cap tao/sua, "
        f"{moved:,} file du chuyen du phong",
        flush=True
    )

# ======================================================
# 5. TAO DATA.YAML
# ======================================================

data_cfg = {
    "path": str(OUTPUT),
    "train": "train/images",
    "val": "valid/images",
    "test": "test/images",
    "nc": 5,
    "names": NEW_NAMES
}

with (OUTPUT / "data.yaml").open(
    "w", encoding="utf-8"
) as f:
    yaml.safe_dump(
        data_cfg,
        f,
        allow_unicode=True,
        sort_keys=False
    )

# ======================================================
# 6. IN THONG KE
# ======================================================

print("\n" + "=" * 50)
print("HOAN THANH RESUME YOLO RAW V1!")
print("=" * 50)

for split, info in summary.items():
    print(f"\n{split.upper()}")
    print("Anh hop le:", info["images"])
    print("Cap anh/nhan tao hoac sua:", info["repaired"])
    print("Anh bo qua:", info["skipped"])
    print("File du chuyen du phong:", info["moved"])

    for i, name in enumerate(NEW_NAMES):
        print(
            f"  {name}: {info['boxes'][i]:,} boxes"
        )

print("\nTong thoi gian:",
      round(time.time() - start_time, 1), "giay")

print("Dataset:", OUTPUT)
print("Config:", OUTPUT / "data.yaml")



RESUME DATASET YOLO RAW V1
Source: F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_raw\source
Output: F:\AI SAMSUNG\FoodVision-AI-V2\datasets\food_raw\processed\raw_5classes_v1

DANG RESUME TRAIN: 35,823 anh nguon
train: 4,000/35,823 | Hop le: 462 | Tao/sua: 162
train: 9,000/35,823 | Hop le: 902 | Tao/sua: 333
train: 14,000/35,823 | Hop le: 1,335 | Tao/sua: 598
train: 15,000/35,823 | Hop le: 1,526 | Tao/sua: 720
train: 17,000/35,823 | Hop le: 1,923 | Tao/sua: 928
train: 17,500/35,823 | Hop le: 2,374 | Tao/sua: 1,106
train: 18,000/35,823 | Hop le: 2,812 | Tao/sua: 1,347
train: 18,500/35,823 | Hop le: 3,307 | Tao/sua: 1,713
